In [4]:
import open3d as o3d
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt

def draw_points_mat(points_list, title = 'point cloud visualize', elev = 45, azim = 120 ):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    color = ['red','orange','yellow','green','blue','black','purple']
    for i in range (len(points_list)):
        ax.scatter(points_list[i][:, 0], points_list[i][:, 1], points_list[i][:, 2], s=1, c = color[i])  # s=1 for small points
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        plt.title(title)
        ax.view_init(elev, azim)
    plt.show()    

    return



def find_plane(points_np, voxel_size=0.001, distance_threshold=0.001):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_np[:, :3])

    pcd = pcd.voxel_down_sample(voxel_size=voxel_size)

    pcd, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    pcd = pcd.select_by_index(ind)

    plane1_model, inliers1 = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=3,
        num_iterations=3000
    )
    plane1_pcd = pcd.select_by_index(inliers1)
    rest_pcd = pcd.select_by_index(inliers1, invert=True)

    plane2_model, inliers2 = rest_pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=3,
        num_iterations=3000
    )
    plane2_pcd = rest_pcd.select_by_index(inliers2)
    rest_pcd = rest_pcd.select_by_index(inliers2, invert=True)

    return plane1_model, plane1_pcd, plane2_model, plane2_pcd, rest_pcd


def get_largest_cluster(pcd, eps=0.02, min_points=10):
    labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=min_points, print_progress=False))
    max_label = labels.max()
    
    if max_label < 0:
        print("未找到有效的连通域（全是噪声）")
        return None

    largest_cluster_id = -1
    max_count = 0

    for i in range(max_label + 1):
        count = np.sum(labels == i)
        if count > max_count:
            max_count = count
            largest_cluster_id = i
            
    print(f"找到 {max_label + 1} 个簇，最大簇包含 {max_count} 个点")

    indices = np.where(labels == largest_cluster_id)[0]
    return pcd.select_by_index(indices)


def plane_from_pcd(plane_pcd):
    pts = np.asarray(plane_pcd.points)
    c = pts.mean(axis=0)
    _, _, vh = np.linalg.svd(pts - c, full_matrices=False)
    n = vh[-1]
    n = n / np.linalg.norm(n)
    return c, n


def plane_basis_from_pcd(plane_pcd):
    c, n = plane_from_pcd(plane_pcd)

    ref = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    bu = np.cross(n, ref)
    bu = bu / np.linalg.norm(bu)
    bv = np.cross(n, bu)
    bv = bv / np.linalg.norm(bv)

    return c, bu, bv, n


def plane_intersection_line(plane1_pcd, plane2_pcd):
    c1, n1 = plane_from_pcd(plane1_pcd)
    c2, n2 = plane_from_pcd(plane2_pcd)

    d = np.cross(n1, n2)
    d_norm = np.linalg.norm(d)
    if d_norm < 1e-10:
        raise ValueError("两个平面几乎平行，无法稳定求交线")
    d = d / d_norm

    A = np.vstack([n1, n2, d])
    b = np.array([n1 @ c1, n2 @ c2, 0.0])
    p0 = np.linalg.solve(A, b)

    return p0, d


def largest_component(mask, min_pixels=30):
    lab, num = ndimage.label(mask)
    if num == 0:
        return np.zeros_like(mask, dtype=bool)
    areas = ndimage.sum(mask, lab, range(1, num + 1))
    idx = np.argmax(areas) + 1
    out = (lab == idx)
    return out if out.sum() >= min_pixels else np.zeros_like(mask, dtype=bool)


def make_pcd(points_np):
    pcd = o3d.geometry.PointCloud()
    if len(points_np) > 0:
        pcd.points = o3d.utility.Vector3dVector(points_np)
    return pcd


def project_points_to_plane(points_or_pcd, plane_pcd):
    """
    支持输入:
      - open3d.geometry.PointCloud
      - ndarray, shape = (N, 3)
    """
    if isinstance(points_or_pcd, o3d.geometry.PointCloud):
        pts = np.asarray(points_or_pcd.points)
    else:
        pts = np.asarray(points_or_pcd)

    if pts.size == 0:
        return np.array([]), np.array([]), np.zeros((0, 3))

    c, u, v, n = plane_basis_from_pcd(plane_pcd)

    rel = pts - c
    du = rel @ u
    dv = rel @ v
    proj_3d = c + np.outer(du, u) + np.outer(dv, v)
    return du, dv, proj_3d


def split_points_by_two_planes_overlap(
    pcd,
    plane1_pcd,
    plane2_pcd,
    dist_thresh=0.002,
    margin=0.0005,
    edge_tol=0.001
):
    """
    core:
        明确属于某一平面的点，只用于 ideal_mask 和侧别判断
    occ:
        用于 actual_mask 的点；包含 core + overlap(edge) 双归属点
    edge:
        歧义点集合本身，便于可视化检查
    """
    pts = np.asarray(pcd.points)

    c1, n1 = plane_from_pcd(plane1_pcd)
    c2, n2 = plane_from_pcd(plane2_pcd)

    d1 = np.abs((pts - c1) @ n1)
    d2 = np.abs((pts - c2) @ n2)

    # 明确属于单一平面的 core 点
    mask1_core = (d1 < dist_thresh) & (d1 + margin < d2)
    mask2_core = (d2 < dist_thresh) & (d2 + margin < d1)

    # 歧义/棱边附近点
    mask_edge = ~(mask1_core | mask2_core)

    # 这些 edge 点只要“也还算靠近某个平面”，就允许进入该平面的占据图
    near_p1 = d1 < (dist_thresh + edge_tol)
    near_p2 = d2 < (dist_thresh + edge_tol)

    mask1_occ = mask1_core | (mask_edge & near_p1)
    mask2_occ = mask2_core | (mask_edge & near_p2)

    pcd1_core = make_pcd(pts[mask1_core])
    pcd2_core = make_pcd(pts[mask2_core])

    pcd1_occ = make_pcd(pts[mask1_occ])
    pcd2_occ = make_pcd(pts[mask2_occ])

    pcd_edge = make_pcd(pts[mask_edge])

    debug = {
        "d1": d1,
        "d2": d2,
        "mask1_core": mask1_core,
        "mask2_core": mask2_core,
        "mask1_occ": mask1_occ,
        "mask2_occ": mask2_occ,
        "mask_edge": mask_edge,
    }

    return pcd1_core, pcd2_core, pcd1_occ, pcd2_occ, pcd_edge, (c1, n1), (c2, n2), debug


def uv_to_defect_mask_dual(
    core_u,
    core_v,
    occ_u,
    occ_v,
    plane_pcd,
    other_plane_pcd,
    grid_res=0.001,
    pad=0.002,
    min_pixels=30,
    side_ignore_band=None,
    line_exclude_band=0.02,
):
    """
    core_u/core_v:
        只用来估计 ideal_mask 范围 + 判断交线哪一侧该保留
    occ_u/occ_v:
        用来构建 actual_mask，允许包含 overlap edge 点
    side_ignore_band:
        侧别判断时，忽略离交线太近的 core 点。默认=1.5*grid_res
    line_exclude_band:
        最终 defect_mask 中，去掉离交线过近的一条带；默认 0，不去掉
        如果你发现两个平面在交线附近各补一层，可设为 grid_res 或 1.5*grid_res
    """
    core_u = np.asarray(core_u)
    core_v = np.asarray(core_v)
    occ_u = np.asarray(occ_u)
    occ_v = np.asarray(occ_v)

    if side_ignore_band is None:
        side_ignore_band = 1.5 * grid_res

    if len(core_u) == 0 or len(occ_u) == 0:
        info = {
            "umin": 0.0,
            "vmin": 0.0,
            "grid_res": grid_res,
            "actual_mask": np.zeros((1, 1), dtype=bool),
            "ideal_mask": np.zeros((1, 1), dtype=bool),
            "defect_mask": np.zeros((1, 1), dtype=bool),
            "line_p0_uv": np.zeros(2),
            "line_p1_uv": np.zeros(2),
            "half_mask": np.zeros((1, 1), dtype=bool),
        }
        return np.zeros((1, 1), dtype=bool), info

    # 用 union 范围建栅格，避免 occ 比 core 更外扩时被截掉
    all_u = np.concatenate([core_u, occ_u])
    all_v = np.concatenate([core_v, occ_v])

    umin = all_u.min() - pad
    umax = all_u.max() + pad
    vmin = all_v.min() - pad
    vmax = all_v.max() + pad

    nx = int(np.ceil((umax - umin) / grid_res)) + 1
    ny = int(np.ceil((vmax - vmin) / grid_res)) + 1

    # 1) actual_mask 只看 occ 点
    actual_mask = np.zeros((ny, nx), dtype=bool)

    ix_occ = np.floor((occ_u - umin) / grid_res).astype(int)
    iy_occ = np.floor((occ_v - vmin) / grid_res).astype(int)
    ix_occ = np.clip(ix_occ, 0, nx - 1)
    iy_occ = np.clip(iy_occ, 0, ny - 1)

    actual_mask[iy_occ, ix_occ] = True
    actual_mask = ndimage.binary_closing(actual_mask, iterations=1)

    # 2) ideal_mask 只看 core 点
    u0, u1 = np.percentile(core_u, [1, 99])
    v0, v1 = np.percentile(core_v, [1, 99])

    ideal_mask = np.zeros((ny, nx), dtype=bool)

    ix0 = int(np.floor((u0 - umin) / grid_res))
    ix1 = int(np.ceil((u1 - umin) / grid_res))
    iy0 = int(np.floor((v0 - vmin) / grid_res))
    iy1 = int(np.ceil((v1 - vmin) / grid_res))

    ix0 = np.clip(ix0, 0, nx - 1)
    ix1 = np.clip(ix1, 0, nx - 1)
    iy0 = np.clip(iy0, 0, ny - 1)
    iy1 = np.clip(iy1, 0, ny - 1)

    ideal_mask[iy0:iy1 + 1, ix0:ix1 + 1] = True

    # 3) 交线裁剪 ideal_mask
    c, bu, bv, n = plane_basis_from_pcd(plane_pcd)
    p0_3d, d_3d = plane_intersection_line(plane_pcd, other_plane_pcd)

    p0_uv = np.array([(p0_3d - c) @ bu, (p0_3d - c) @ bv])
    p1_uv = np.array([(p0_3d + d_3d - c) @ bu, (p0_3d + d_3d - c) @ bv])

    du_line = p1_uv[0] - p0_uv[0]
    dv_line = p1_uv[1] - p0_uv[1]

    # 侧别判断只看 core 点，并忽略贴交线太近的点
    side_core = du_line * (core_v - p0_uv[1]) - dv_line * (core_u - p0_uv[0])
    valid_side = np.abs(side_core) > side_ignore_band

    if np.any(valid_side):
        side_used = side_core[valid_side]
    else:
        # 万一 core 点几乎都贴着交线，就退化为全部 core 点投票
        side_used = side_core

    keep_positive = np.sum(side_used >= 0) >= np.sum(side_used <= 0)

    uu = umin + (np.arange(nx) + 0.5) * grid_res
    vv = vmin + (np.arange(ny) + 0.5) * grid_res
    UU, VV = np.meshgrid(uu, vv)

    side_grid = du_line * (VV - p0_uv[1]) - dv_line * (UU - p0_uv[0])

    if keep_positive:
        half_mask = side_grid >= -grid_res
    else:
        half_mask = side_grid <= grid_res

    ideal_mask = ideal_mask & half_mask

    # 4) 缺陷 = 理想 - 实际
    defect_mask = ideal_mask & (~actual_mask)

    # 可选：去掉交线附近一条带，避免两个平面都在棱边补点
    if line_exclude_band > 0:
        defect_mask = defect_mask & (np.abs(side_grid) > line_exclude_band)

    defect_mask = largest_component(defect_mask, min_pixels=min_pixels)

    info = {
        "umin": umin,
        "vmin": vmin,
        "grid_res": grid_res,
        "actual_mask": actual_mask,
        "ideal_mask": ideal_mask,
        "defect_mask": defect_mask,
        "line_p0_uv": p0_uv,
        "line_p1_uv": p1_uv,
        "half_mask": half_mask,
    }
    return defect_mask, info


def defect_mask_to_3d(defect_mask, info, plane_pcd):
    c, bu, bv, n = plane_basis_from_pcd(plane_pcd)

    ys, xs = np.where(defect_mask)
    if len(xs) == 0:
        return o3d.geometry.PointCloud()

    u = info["umin"] + (xs + 0.5) * info["grid_res"]
    v = info["vmin"] + (ys + 0.5) * info["grid_res"]

    pts3d = c + np.outer(u, bu) + np.outer(v, bv)

    pcd_defect = o3d.geometry.PointCloud()
    pcd_defect.points = o3d.utility.Vector3dVector(pts3d)
    return pcd_defect

In [5]:
####### run code
pcd_raw = o3d.io.read_point_cloud("E:/HKUSTGZ/AAM/construction/data/completion_result/fused2.pcd")
pcd_raw = pcd_raw.voxel_down_sample(voxel_size=0.001)

pcd = get_largest_cluster(pcd_raw)

plane1_model, plane1_pcd, plane2_model, plane2_pcd, rest_pcd = find_plane(
    np.asarray(pcd.points),
    voxel_size=0.003,
    distance_threshold=0.0015
)

print("Plane 1:", plane1_model)
print("Plane 2:", plane2_model)

# 这里换成 overlap 版本
pcd1_core, pcd2_core, pcd1_occ, pcd2_occ, pcd_edge, plane1, plane2, debug = split_points_by_two_planes_overlap(
    pcd,
    plane1_pcd,
    plane2_pcd,
    dist_thresh=0.002,
    margin=0.0005,
    edge_tol=0.0010,   # 这里可以调大一点，比如 0.001 ~ 0.0015
)

# plane1: core 点投影
u1_core, v1_core, proj1_core_3d = project_points_to_plane(pcd1_core, plane1_pcd)
# plane1: 占据图点投影（core + overlap）
u1_occ, v1_occ, proj1_occ_3d = project_points_to_plane(pcd1_occ, plane1_pcd)

# plane2: core 点投影
u2_core, v2_core, proj2_core_3d = project_points_to_plane(pcd2_core, plane2_pcd)
# plane2: 占据图点投影（core + overlap）
u2_occ, v2_occ, proj2_occ_3d = project_points_to_plane(pcd2_occ, plane2_pcd)

# 返回两个平面的缺陷 mask
defect_mask1, info1 = uv_to_defect_mask_dual(
    core_u=u1_core,
    core_v=v1_core,
    occ_u=u1_occ,
    occ_v=v1_occ,
    plane_pcd=plane1_pcd,
    other_plane_pcd=plane2_pcd,
    grid_res=0.001,
    pad=0.001,
    min_pixels=30,
    side_ignore_band=0.0015,
    line_exclude_band=0.0,   # 若棱边重复补点，可改成 0.001
)

defect_mask2, info2 = uv_to_defect_mask_dual(
    core_u=u2_core,
    core_v=v2_core,
    occ_u=u2_occ,
    occ_v=v2_occ,
    plane_pcd=plane2_pcd,
    other_plane_pcd=plane1_pcd,
    grid_res=0.001,
    pad=0.001,
    min_pixels=30,
    side_ignore_band=0.0015,
    line_exclude_band=0.0,   # 若棱边重复补点，可改成 0.001
)

# 重投影回 3D
defect_pcd1 = defect_mask_to_3d(defect_mask1, info1, plane1_pcd)
defect_pcd2 = defect_mask_to_3d(defect_mask2, info2, plane2_pcd)

# 合并补全点云
pts1 = np.asarray(defect_pcd1.points)
pts2 = np.asarray(defect_pcd2.points)

defect_all = o3d.geometry.PointCloud()
if len(pts1) + len(pts2) > 0:
    defect_all.points = o3d.utility.Vector3dVector(np.vstack([pts1, pts2]))

    # 建议做一次轻微去重，避免两平面补点过近
    defect_all = defect_all.voxel_down_sample(voxel_size=0.0008)

# 可视化检查
o3d.visualization.draw_geometries([
    pcd.paint_uniform_color([0.7, 0.7, 0.7]),
    pcd1_core.paint_uniform_color([1, 0, 0]),      # 红：plane1 core
    pcd2_core.paint_uniform_color([0, 0, 1]),      # 蓝：plane2 core
    pcd_edge.paint_uniform_color([0, 1, 0]),       # 绿：edge/ambiguity
    defect_all.paint_uniform_color([1, 1, 0]),     # 黄：补点
])

找到 2 个簇，最大簇包含 59591 个点
Plane 1: [-0.09291408 -0.26303882  0.96030076 -0.10236322]
Plane 2: [ 0.97733626 -0.20764981  0.04117501 -0.37915128]
